# Full-Body Mesh Repair (FLAME + Geometric)

Repairs FBX files with:
- **Head/scalp holes** → filled using FLAME 3D face model as reference surface
- **Body holes** → geometric triangulation fill
- **Non-watertight mesh** → closed and normals fixed

All model files and test FBX are included in the repo — no downloads needed.

In [ ]:
%%capture
!pip install torch numpy trimesh scipy tqdm
!apt-get update -qq && apt-get install -y -qq blender > /dev/null 2>&1

In [ ]:
import os
if not os.path.exists('/content/demo2026'):
    !git clone https://github.com/huijieqi/demo2026.git /content/demo2026
os.chdir('/content/demo2026')
import sys
sys.path.insert(0, '/content/demo2026')

In [ ]:
import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

import trimesh
print(f'Trimesh: {trimesh.__version__}')

import subprocess
result = subprocess.run(['blender', '--version'], capture_output=True, text=True)
print(f'Blender: {result.stdout.strip().split(chr(10))[0]}')

# Verify files exist
assert os.path.exists('input/liukui.fbx'), 'Missing input/liukui.fbx'
assert os.path.exists('models/generic_model.pkl'), 'Missing models/generic_model.pkl'
print('\nAll files ready!')

## Run Full-Body Repair

Pipeline:
1. Load FBX (via Blender → OBJ conversion)
2. Basic cleanup (degenerate faces, duplicate faces, normals)
3. Detect all holes (boundary loops)
4. Classify holes: head region vs body
5. Head holes → FLAME-guided fill (anatomically correct scalp)
6. Body holes → geometric fan triangulation
7. Final repair pass + smoothing
8. Export as FBX

In [ ]:
from flame_repair.pipeline import MeshRepairPipeline

pipeline = MeshRepairPipeline(
    flame_model_path='models/generic_model.pkl',
    device='auto',
)

result_mesh = pipeline.run(
    input_path='input/liukui.fbx',
    output_path='output/repaired_liukui.fbx',
    mode='full_body_repair',
    smooth_iters=2,
)

## Visualize Before / After

In [ ]:
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
import numpy as np
from flame_repair.mesh_io import load_mesh

original = load_mesh('input/liukui.fbx')

def plot_comparison(mesh_a, mesh_b, title_a='Original', title_b='Repaired', max_faces=8000):
    fig, axes = plt.subplots(1, 2, figsize=(16, 8), subplot_kw={'projection': '3d'})

    for ax, mesh, title in [(axes[0], mesh_a, title_a), (axes[1], mesh_b, title_b)]:
        verts = mesh.vertices
        faces = mesh.faces
        if len(faces) > max_faces:
            idx = np.random.choice(len(faces), max_faces, replace=False)
            faces = faces[idx]
        ax.plot_trisurf(verts[:, 0], verts[:, 1], verts[:, 2],
                        triangles=faces, alpha=0.6, edgecolor='gray', linewidth=0.05)
        ax.set_title(f'{title}\n{len(mesh.vertices)} verts, {len(mesh.faces)} faces\nWatertight: {mesh.is_watertight}')

    plt.tight_layout()
    plt.show()

plot_comparison(original, result_mesh)

## Download Repaired File

In [ ]:
from google.colab import files
import glob

for f in glob.glob('output/*'):
    print(f'Downloading: {f}')
    files.download(f)

---

## (Optional) Use Other Modes

```python
# repair_only - just basic mesh repair, no hole detection
pipeline.run('input/liukui.fbx', 'output/basic.fbx', mode='repair_only')

# flame_fit - fit FLAME model to head, output FLAME topology (head only)
pipeline.run('input/liukui.fbx', 'output/flame.fbx', mode='flame_fit')

# flame_blend - blend original with FLAME fitting (head only)
pipeline.run('input/liukui.fbx', 'output/blend.fbx', mode='flame_blend')
```